# Inititial Pose Estimation of a Banana

In [ ]:
# imports
import copy
import cv2
import matplotlib.pyplot as plt
import numpy as np
import os
import open3d as o3d
from open3d.web_visualizer import draw as web_draw
import sys
from ultralytics import YOLO

sys.path.append(os.path.abspath('../src'))
from camera import Camera
from evaluation import Evaluation
from paths import PathManager
from yolo11 import predict_segment

In [ ]:
# Settings for this notebook
index = 50
# 16
web_visualization = False

In [ ]:
# draw function: depending on web_visualization
def draw(point_clouds):
    global web_visualization
    if web_visualization:
        web_draw(point_clouds)
    else:
        o3d.visualization.draw_geometries(point_clouds)

In [ ]:
searched_item = 'banana'
pm = PathManager(searched_item)

In [ ]:
rgb_image = np.load(pm.get_rgb_image_path(index))
bgr_image = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2BGR)
plt.imshow(bgr_image)
plt.title("Original Image")
plt.axis(False)
plt.show()

In [ ]:
depth_image = np.load(pm.get_depth_image_path(index))
plt.imshow(depth_image, cmap="viridis")  # alternatives: 'gray' or 'plasma'
plt.title("Depth Image")
plt.axis(False)
plt.colorbar()  # show colorbar
plt.show()

In [ ]:
pts, seg_image = predict_segment(rgb_image, searched_item)

In [ ]:
plt.imshow(cv2.cvtColor(seg_image, cv2.COLOR_BGR2RGB))  # Convert BGR to RGB for correct color representation
plt.title("Segmantation")
plt.axis(False)
plt.show()

In [ ]:
# Create a mask initialized with zeros (same shape as depth image)
mask = np.zeros_like(depth_image)
cv2.fillPoly(mask, [pts], color=1)  # Set inside polygon to 1
masked_depth_image = depth_image * mask

In [ ]:
# Show depth image
plt.imshow(masked_depth_image, cmap="viridis")
plt.title("masked depth image")
plt.axis(False)
plt.colorbar()
plt.show()

In [ ]:
cam = Camera.from_yaml("../data/camera_intrinsics.yaml", "D435i")

In [ ]:
def get_scene_point_cloud(depth_img, d_width, d_height, cam_params):
    (fx, fy, cx, cy) = cam_params
    depth_img = o3d.geometry.Image(depth_img)
    cam_intrinsics = o3d.camera.PinholeCameraIntrinsic(
        width=d_width,
        height=d_height,
        fx=fx,
        fy=fy,
        cx=cx,
        cy=cy)
    pcd = o3d.geometry.PointCloud.create_from_depth_image(
        depth=depth_img,
        intrinsic=cam_intrinsics,
        extrinsic=np.eye(4)
    )
    return pcd

In [ ]:
pcd = get_scene_point_cloud(depth_image, cam.width, cam.height, cam.get_camera_intrinsics())

In [ ]:
draw([pcd])

In [ ]:
cropped_pcd = get_scene_point_cloud(masked_depth_image, cam.width, cam.height, cam.get_camera_intrinsics())

In [ ]:
draw([cropped_pcd])

In [ ]:
model = o3d.io.read_triangle_mesh(pm.get_model_path())

In [ ]:
model.compute_vertex_normals()
draw([model])

In [ ]:
model_pcd = model.sample_points_poisson_disk(number_of_points=10000)

In [ ]:
draw([model_pcd])

In [ ]:
# create copy of cropped_pcd
filtered_pcd = copy.deepcopy(cropped_pcd)

In [ ]:
# remove outliers
filtered_pcd, ind = filtered_pcd.remove_statistical_outlier(nb_neighbors=50, std_ratio=0.7)

In [ ]:
# estimate normals
filtered_pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))

In [ ]:
draw([filtered_pcd])

In [ ]:
# show cropped pcd in original pcd
pcd_black = copy.deepcopy(pcd).paint_uniform_color([0,0,0])
draw([pcd_black, filtered_pcd])

In [ ]:
print(f"Model center:       {model_pcd.get_center()}")
print(f"Point cloud center: {filtered_pcd.get_center()}")

In [ ]:
T = np.eye(4)
T[:3, 3] = filtered_pcd.get_center() - model_pcd.get_center()
print(f"Translation: {T[:3, 3]}")

In [ ]:
# transform with calculatet T
transformed_model = copy.deepcopy(model_pcd).transform(T)

In [ ]:
def preprocess_point_cloud(pcd, voxel_size):
    pcd_down = pcd.voxel_down_sample(voxel_size)
    pcd_down.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size*2, max_nn=30)
    )
    fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        pcd_down,
        search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size*5, max_nn=100)
    )
    return pcd_down, fpfh

In [ ]:
voxel_size = 0.005 # in meters -> 5mm
scene_down, scene_fpfh = preprocess_point_cloud(filtered_pcd, voxel_size)
model_down, model_fpfh = preprocess_point_cloud(transformed_model, voxel_size)

print(scene_down, scene_fpfh)
print(model_down, model_fpfh)

In [ ]:
result_ransac = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    model_down,             # source
    scene_down,             # target
    model_fpfh,             # source feature
    scene_fpfh,             # target feature
    mutual_filter=False,    # mutual_filter
    max_correspondence_distance=voxel_size*1.5,
    estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
    ransac_n=4,
    checkers=[
        o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.9),
        o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(voxel_size*1.5),
    ],
    criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(4000000, 500)
)
print(result_ransac)

In [ ]:
draw([scene_down, model_down])

In [ ]:
source = copy.deepcopy(model_down)
target = copy.deepcopy(scene_down)
threshold = voxel_size

In [ ]:
reg_p2p = o3d.pipelines.registration.registration_icp(
    source=source, 
    target=target, 
    max_correspondence_distance=threshold,
    init=result_ransac.transformation,
    estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(),
    criteria=o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=100),
)

print("Transformation matrix:")
print(reg_p2p.transformation)

In [ ]:
transformed_source = copy.deepcopy(source).transform(reg_p2p.transformation)
draw([transformed_source, target])

In [ ]:
T_gt = np.genfromtxt(pm.get_orientation_path(index))
print("Ground Truth:\n", T_gt)

In [ ]:
eval = Evaluation(source, target, threshold, reg_p2p.transformation, T_gt)
print(eval)